In [21]:
import pandas as pd
from pathlib import Path
import duckdb
import sys
from openpyxl.writer.excel import ExcelWriter

In [2]:
file_path = Path(r"SHOPEE CODE.xlsm")

In [3]:
combo_df_rn = {
    'Tên sản phẩm': 'combo_name',
    'Tên phân loại hàng': 'combo_variant',
    'Giá ưu đãi': 'total_price',
    'Số loại sp': 'combo_product_count',
    'Tên sản phẩm khi tách': 'product_name',
    'Giá ưu đãi khi tách': 'product_price',
    'số lượng khi tách': 'product_quantity',
}

code_df_rn = {
    'Mã CODE': 'product_code',
    'Tên SP': 'product_name'
}

def read_sheet(path, sn: str, rn_dict: dict):
    old_cols = [col for col in rn_dict.keys()]
    new_cols = [col for col in rn_dict.values()]
    df = pd.read_excel(path, sheet_name=sn, usecols=old_cols)
    df_rename = df.rename(columns=rn_dict)
    df_final = df_rename[new_cols]
    return df_final

combo_df = read_sheet(file_path, "DLCOMBO", combo_df_rn)
code_df = read_sheet(file_path, "CODE", code_df_rn)

In [4]:
product = duckdb.sql("""
    SELECT DISTINCT
    "product_code",
    "product_name"
    FROM code_df
""")

dlcombo_extract = duckdb.sql("""
    SELECT
        cb.combo_name,
        NULLIF(cb.combo_variant, '0') as combo_variant,
        pl.product_code,
        pl.product_name,
        cb.combo_product_count,
        cb.product_price,
        cb.product_quantity,
        cb.product_price * cb.product_quantity as total_byrow,
        cb.total_price as total_value,

        SUM(cb.product_price * cb.product_quantity)
            OVER (
            PARTITION BY cb.combo_name, cb.combo_variant, cb.total_price
            ) as total_recalculated,

        COUNT(cb.combo_name)
            OVER(
            PARTITION BY cb.combo_name, cb.combo_variant, cb.total_price
            ) AS product_count
    FROM combo_df cb
    JOIN product pl USING (product_name)
    -- Lọc bỏ các product_code không khớp với combo và các mã sản phẩm = 0
    WHERE
        pl.product_code IS NOT NULL
        AND pl.product_code <> 0
    ORDER BY combo_name, total_value
""").to_df()

# Trích xuất các combo có trong VBA
combo = duckdb.sql("""
    SELECT
        row_number() OVER() - 1 as combo_key,
        combo_name,
        combo_variant,
        total_recalculated
    FROM (
        SELECT DISTINCT combo_name, combo_variant, total_recalculated
        FROM dlcombo_extract
        ORDER BY 1, 2, 3
    )
""").to_df()

# Trích xuất danh sách sản phẩm
products = duckdb.sql("""
    SELECT DISTINCT
    product_code,
    product_name
    FROM dlcombo_extract
    ORDER BY product_code
""").to_df()

# Bảng chi tiết: Ánh xạ sản phẩm vào từng combo_key
combo_details = duckdb.sql("""
    SELECT
        c.combo_key,
        cd.product_code,
        cd.product_name,
        cd.product_price,
        cd.product_quantity
    FROM dlcombo_extract cd
    LEFT JOIN combo c
        ON cd.combo_name IS NOT DISTINCT FROM c.combo_name
        AND cd.combo_variant IS NOT DISTINCT FROM c.combo_variant
        AND cd.total_recalculated = c.total_recalculated
    ORDER BY c.combo_key, cd.product_code
""").to_df()

In [5]:
# Các combo bị điền sai thông tin về tổng số lượng sản phẩm
# Các dòng này sẽ bị drop đi vì cài đặt sai
filter = dlcombo_extract['combo_product_count'] != dlcombo_extract['product_count']
false_product = dlcombo_extract[filter].loc[:, ['combo_name', 'combo_variant', 'total_value', 'product_code', 'product_price', 'combo_product_count', 'product_count']]

In [61]:
# Tạo bảng combo - product code hoàn chỉnh
new_master_data = duckdb.sql("""
    SELECT
    combo_name,
    combo_variant,
    product_code,
    product_name,
    product_quantity,
    product_price,
    total_recalculated
    FROM combo_details
    JOIN combo USING (combo_key)
""").to_df()

<h3> Tải file control

In [68]:
order_df = pd.read_excel('đơn tải sàn.xlsx')
order_df = order_df.rename(columns=rename_dict)
join_df = duckdb.sql("""
    WITH order_processed AS (
        SELECT
            order_id,
            product_name AS combo_name,
            variation_name AS combo_variant,
            quantity,
            deal_price,
            SUM(deal_price * quantity) OVER (PARTITION BY od.order_id) as total_recalculated
        FROM order_df od
    )
    SELECT *, ms.total_recalculated FROM order_processed op
    LEFT JOIN new_master_data ms USING (combo_name, combo_variant, total_recalculated)
    WHERE product_code IS NULL
""").to_df()
join_df

,order_id,combo_name,combo_variant,quantity,deal_price,total_recalculated,product_code,product_name,product_quantity,product_price,total_recalculated_1
0,260506NM502QBV,Lá kim ăn liền O’Food cho trẻ em – Lốc 3 gói –...,NaN,1,38000.0,38000.0,NaN,None,<NA>,NaN,NaN
1,260506NKVN3ESN,"Sốt ướp thịt Hàn Quốc OFood gói 80g, giúp thị ...",Vị truyền thống,2,11759.0,23518.0,NaN,None,<NA>,NaN,NaN
2,260506NU2Y3WUK,"Sốt ướp thịt Hàn Quốc OFood gói 80g, giúp thị ...",Vị truyền thống,1,0.0,210000.0,NaN,None,<NA>,NaN,NaN
3,260506NKSA9GGU,"[O'Food] Bột chiên giòn gói 100g, 500g",500g,1,30000.0,60000.0,NaN,None,<NA>,NaN,NaN
4,260506NTTRUP46,"[O'Food] Rong biển giòn trộn khô gà, hải sản, ...",trộn hải sản 40g,1,41000.0,157000.0,NaN,None,<NA>,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
742,260506NVK54P7W,"[O'Food] Rong biển giòn trộn khô gà, hải sản, ...",trộn khô gà 40g,1,41000.0,160000.0,NaN,None,<NA>,NaN,NaN
743,260506NVK54P7W,"[O'Food] Rong biển giòn trộn khô gà, hải sản, ...","óc chó,hạnh nhân 40g",1,41000.0,160000.0,NaN,None,<NA>,NaN,NaN
744,260505KXWUHVAV,[PHIÊN BẢN 24 GÓI] Rong Biển Lá Kim O’Food Ô L...,NaN,4,145000.0,580000.0,NaN,None,<NA>,NaN,NaN
745,260505M1EADTUV,[PHIÊN BẢN 24 GÓI] Rong Biển Lá Kim O’Food Ô L...,NaN,4,145000.0,580000.0,NaN,None,<NA>,NaN,NaN


In [39]:
mapping_df = pd.read_excel('shopee_control_file.xlsx', sheet_name='column_mapping')

In [40]:
rename_dict = dict(zip(mapping_df['raw_name'], mapping_df['sys_name']))

<h3> Tải file hoá đơn